In [ ]:
!pip install rdkit
!pip install selfies
import pandas as pd
import selfies as sf
import random
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors, rdMolDescriptors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 18.5 MB/s eta 0:00:00


The idea of SELFIES token mutation is adapted from:
Nigam, A. et al., A Beyond Generative Models: Superfast Traversal, Optimization, Novelty, Exploration and Discovery (STONED) Algorithm for Molecules Using SELFIES. Chem. Sci. 2021, 12 (20), 7079–7090., DOI: 10.1039/D1SC00231G

In [ ]:
def mutate_selfies(s, num_mutations=1):
    tokens = list(sf.split_selfies(s))
    for _ in range(num_mutations):
        if not tokens:
            return s
        i = random.randint(0, len(tokens) - 1)
        alphabet = list(sf.get_alphabet_from_selfies([s]))
        tokens[i] = random.choice(alphabet)
    return ''.join(tokens)


SARA constraints were based on: Iwase, M. et al., Development of Digital Oil for Heavy Crude Oil: Molecular Model and Molecular Dynamics Simulations. Energy Fuels 2018, 32 (3), 2781–2792. DOI: 10.1021/acs.energyfuels.7b02881.

In [ ]:
def has_valid_fused_rings(mol):
    # Obtain ring information for the molecule
    ri = mol.GetRingInfo()
    atom_rings = ri.AtomRings()

    # Iterate over all unique pairs of rings
    for i in range(len(atom_rings)):
        for j in range(i + 1, len(atom_rings)):
            ring1 = set(atom_rings[i])
            ring2 = set(atom_rings[j])
            common_atoms = ring1 & ring2

            # Consider only rings that share atoms (i.e., fused rings)
            if len(common_atoms) > 0:
                # Valid fused rings must share exactly two atoms
                if len(common_atoms) != 2:
                    return False  # Too many or too few shared atoms → invalid fusion

                # The shared atoms must be directly bonded in the molecular graph
                common_atoms = list(common_atoms)
                if not mol.GetBondBetweenAtoms(common_atoms[0], common_atoms[1]):
                    return False  # Shared atoms are not adjacent → not a proper fused system

    return True  # All fused ring systems are valid



In [ ]:
def satisfies_saturates_constraints(mol):
    # Check that all rings contain 5–6 carbon atoms
    for ring in atom_rings:
        ring_atoms = [mol.GetAtomWithIdx(idx) for idx in ring]
        num_carbons = sum(1 for atom in ring_atoms if atom.GetAtomicNum() == 6)
        if not (5 <= num_carbons <= 6):
            return False

    # Limit the number of aliphatic (non-aromatic) rings
    if num_aliphatic > 3:
        return False

    # Allow only hydrogen and carbon atoms
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() not in (1, 6):
            return False

    # Exclude heavy, highly aromatic, polycyclic structures
    if num_aromatic > 1 and mol_wt >= 400 and num_rings >= 4:
        return False

    # Exclude molecules already present in the reference saturates dataset
    if Chem.MolToSmiles(mol) in saturated_old.iloc[:, 1].values:
        return False

    return True


def satisfies_sara_constraints(mol, sara_type):
    # Reject invalid molecules
    if mol is None:
        return False

    # Compute basic molecular descriptors
    num_rings = rdMolDescriptors.CalcNumRings(mol)
    num_aromatic = rdMolDescriptors.CalcNumAromaticRings(mol)
    num_aliphatic = num_rings - num_aromatic
    mol_wt = Descriptors.MolWt(mol)

    # Retrieve ring topology
    ring_info = mol.GetRingInfo()
    atom_rings = ring_info.AtomRings()

    # Restrict allowed elements (H, C, N, O, S)
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() not in (1, 6, 7, 8, 16):
            return False

    # Ensure all rings contain 5–6 carbon atoms
    for ring in atom_rings:
        ring_atoms = [mol.GetAtomWithIdx(idx) for idx in ring]
        num_carbons = sum(1 for atom in ring_atoms if atom.GetAtomicNum() == 6)
        if not (5 <= num_carbons <= 6):
            return False

    # Disallow triple bonds
    for bond in mol.GetBonds():
        if bond.GetBondType() == Chem.rdchem.BondType.TRIPLE:
            return False

    # Disallow C=O and C=N double bonds
    for bond in mol.GetBonds():
        if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE:
            begin = bond.GetBeginAtom()
            end = bond.GetEndAtom()
            atoms = {begin.GetAtomicNum(), end.GetAtomicNum()}
            if atoms == {6, 8} or atoms == {6, 7}:
                return False

    # Disallow non-aromatic C=C double bonds
    double_bond = Chem.MolFromSmarts("[C]=[C]")
    for match in mol.GetSubstructMatches(double_bond):
        atoms = [mol.GetAtomWithIdx(i) for i in match]
        if not all(atom.GetIsAromatic() for atom in atoms):
            return False

    # Exclude specific forbidden fused aromatic motifs
    fused_arom = Chem.MolFromSmarts("C1=CC2=CC(=C1)CCC2")
    if mol.HasSubstructMatch(fused_arom):
        return False

    fused_arom = Chem.MolFromSmarts("C1=C2C=CC(=C1)CC2")
    if mol.HasSubstructMatch(fused_arom):
        return False

    # Validate fused ring topology
    if not has_valid_fused_rings(mol):
        return False

    # Restrict sulfur valence
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 16 and atom.GetExplicitValence() > 2:
            return False

    # Disallow chemically implausible heteroatom–heteroatom bonds
    for bond in mol.GetBonds():
        atom1 = bond.GetBeginAtom().GetAtomicNum()
        atom2 = bond.GetEndAtom().GetAtomicNum()
        pair = tuple(sorted((atom1, atom2)))

        forbidden_bonds = {
            (7, 7),    # N–N
            (7, 8),    # N–O
            (8, 8),    # O–O
            (8, 16),   # S–O
            (16, 16),  # S–S
            (7, 16)    # N–S
        }

        if pair in forbidden_bonds:
            return False

    # Apply SARA-class-specific constraints
    if sara_type == 'Saturates':
        return satisfies_saturates_constraints(mol)

    elif sara_type == 'Aromatics':
        return (1 <= num_aromatic <= 6) and (400 < mol_wt < 600) and (0 <= num_aliphatic <= 6)

    elif sara_type == 'Resins':
        return (2 <= num_aromatic <= 8) and (0 <= num_aliphatic <= 8) and (400 < mol_wt < 1000)

    elif sara_type == 'Asphaltenes':
        return (num_aromatic >= 3) and (num_rings >= 5) and (mol_wt >= 800)

    return False



In [ ]:
max_attempts = 5000
attempts = 0

# Fix random seed for reproducibility
random.seed(50)

# Initial SMILES templates for each SARA fraction
base_smiles = {
    'Saturates': [
        'CCCCC1CCC2CCCC3CCCC1C32',
        'CCCCCCC1CCC2CCC3CCCCC3C2C1',
        'CC(C)CC(C)CC(C)CC(C)c1ccc2c(c1)CCCC2',
        'CCC(CC(C)CC(C)C)CC1CCCc2ccccc21',
        'CC(C)CC(C)C1CCC2CCCc3cccc1c32',
        'CCC(CCC(C)CC(CC)CC1CC2CCCc3ccc4c(c32)C1CCC4)CC(C)C',
        'CC(C)CC(C)CC(C)CC(C)C1CCc2ccc3c4c2C1CCC4CCC3',
        'CC(C)CC(C)CCCC(C)CCc1cc2c3c(c1)CCCC3CCC2',
        'CCCCC(C)CC(CC)CC(C)c1ccc2c(c1)CCCC2',
        'CCCC(C)CCC(C)Cc1ccc2c(c1)CCCC2',
    ],  # 10 SMILES

    'Aromatics': [
        'Cc1ccc2c3c(cc(CCc4cc(C)c5ccccc5c4)c2c1)CC(C)C(C)C3',
        'CCc1ccc2cc3c(ccc4c5c6c(c7c(cc6cc43)CCCC7)C(C)C(CC)C5C)cc2c1',
        'Cc1c2ccccc2cc2c3c4c(c5c(cc4cc12)C(C)CC(C)C5)CC(C)C3',
        'CCc1cccc2c(CCCCCCC3CCc4cc5c(O)cc(C)cc5cc4C3CC)c3ccccc3cc12',
        'CC1CCCc2cc3oc4cc5c(cc4c3cc21)C(C)C(C)C(C)C5',
        'CCCCCCC1CCC(Cc2cccc3cc(-c4cc(CC(C)C)ccc4CCC)ccc23)c2ccc3c(c21)CC(C)C(CCC)C3',
        'CCCc1ccc2c3ccc(CCc4cc(O)c5cc6cc(CCCc7cccc(CC(C)C)c7)c(C)cc6cc5c4)cc3c3cc4ccc5c(c4cc3c2c1)CCC(CC)C5',
        'CCCC1C(C)Cc2c3c(c4c(c2C1C)CCCC4)C(C)CCC3',
        'CCCCCCC1Cc2c(cc(CCCc3ccccc3)c3ccc(O)cc23)C(CC)C1',
        'CCCC1c2cccc3c2c(cc2c4cc5c(cc4c(CC)cc32)C(C)CCC5)CC1C'
    ],

    'Resins': [
        'CCc1ccc2c3c(cc(O)c2c1)C(C)CCC3',
        'CCCCCCCC1Cc2c3c(cc4cc5c(O)cc(CCOC)cc5c(c24)C1)C(C)CC(C)C3',
        'COCCc1ccc2oc3ccccc3c2c1',
        'CCCCCCCCC1CCCc2c1cc1c3c(sc1c2CCOC)C(CCCCC)CC(CCCOCCCC1CCCc2c1c(O)c1c(CC)c4c(c5c1c2CC(CC(C)C)C5)CC(CCOC)C(C)C4CCOC)C3',
        'CSCCCC1CCc2cc3sc4ccccc4c3cc21',
        'CCCCCCC1CCCc2c1cc1cc3c(O)cccc3c(CC)c1c2CCCCOCCCc1cc(CCCC)cc2c1sc1ccccc12',
        'CCCCCCCc1cccc8c1ccc7c(CCCOCCCCc5c2CCCCc2c3c6c(CCC3CCCOC)c4c(O)cccc4c(CC)c56)cc(CCCC)cc78',
        'CCCc1ccc(CCCCSCCCCCc2cccc3c2oc2c(CCC)ccc(C)c23)cc1',
        'CCC(C)Cc1ccc(C)c2cc(Cc3ccc(C)c4c3sc3cc(C)ccc34)ccc12',
        'CCCCCCc1cc2oc3c(Cc4cc5c(CCSCC)cccc5cc4C)cccc3c2cc1CC'
    ],

    'Asphaltenes': [
        'CCCCCSCCCc1cccc2cc3c(cc12)c1ccc2cc(CC)c4cc5c6cc(CCCC(C)CC)c(C)cc6cc6c7cc(CCCSCCCC)cc8cc3c3c1c2c4c(c56)c3c87',
        'CCCCCSCCCCCc1ncc2cc3c(CC(C)CC=N)ccc4c5cc6ccc7c(CC)c8cc(CCCSCCCC)ccc8cc7c6c6cc7cc(CC(C)CC)cc8c1c2c(c34)c(c78)c56',
        'CCCCc1cc2c(CC)ccc3c2c2c1cc1ccc4c5c6c(CCCC)cncc6c6cc(CCCSCC)c7cccc8c7c6c5c5c8c6ccc7c8c(CC)cccc8c(C)c8c7c6c6c(c38)c2c1c4c56',
        'CCCCCOCCCCCCCCCC1Cc2c3ccc(CCCSCC)cc3cc3c2c2c4c5c6c(cc7nccc8cc9cc(CCCCC)c3c4c9c6c87)CC3C(CCSCC)CCC(C21)C53',
        'CCCCCCc1c(C)ccc2c1ccc1c3c4c5c6c(cc(CCSCC)c7cc(CC(C)C)c8cc9cc%10ccnc%11c%10c%10c9c(c8c76)C5C(C3)C%10C(CC(C)CC)C%11)cc4cc12',
        'CCCCCCCCc1cc2c(cc1CCCCC)c1c3nc4ccc5cccc(CCCCCSCCCCC)c5c4c4cc5ccc(CCSCC)cc5c(c5cc6c7cc(CC)cc(CC)c7cc7ccc2c(c76)c51)c43',
        'CCCCc1cc2c(CC)ccc3c2c2c1cc1ccc4c5c6c(CCCP)cncc6c6cc(CCCSCC)c7cccc8c7c6c5c5c8c6ccc7c8c(CC)cccc8c(C)c8c7c6c6c(c38)c2c1c4c56',
        'CCCCCCCCCc1cc2c(cc1C)cc1c3cc(CCCSCCCC)cc4cc5c6cc7cccc(CCCSCCCCC)c7cc6c6ccc7cc(CC)c8cc2c1c1c8c7c6c5c1c43',
        'CCCCCCCCCCCCCCCCPCCc1cc2c3c4cc5ccccc5cc4c4c(C)ccc5cc(C)c(c6c7ccnc8c(CCC)c(CCCCSCCCCC)c9ccc1c(c9c87)c26)c3c54',
        'CCCCCCCCc1cc2c(cc1CCCCC)c1c3nc4ccc5cccc(CCCCCSCCCCC)c5c4c4cc5ccc(CCSCC)cc5c(c5cc6c7cc(CC)cc(CC)c7cc7ccc2c(c76)c51)c43'
    ]
}

# Convert SMILES strings to SELFIES representation
base_selfies = {
    sara_type: [sf.encoder(smi) for smi in smiles_list]
    for sara_type, smiles_list in base_smiles.items()
}

# Initialize dictionary to store generated SMILES
generated_mols = {k: [] for k in base_smiles}

# Generate new molecules by SELFIES mutation
for sara_type in ['Saturates', 'Aromatics', 'Resins', 'Asphaltenes']:
    while len(set(generated_mols[sara_type])) < 90:
        # Randomly select a base SELFIES string
        selfie = random.choice(base_selfies[sara_type])

        # Apply a random mutation in SELFIES space
        mutated = mutate_selfies(selfie)

        # Decode mutated SELFIES back to SMILES
        smiles = sf.decoder(mutated)

        # Convert SMILES to RDKit molecule
        mol = Chem.MolFromSmiles(smiles)

        # Retain molecules that are chemically valid and satisfy constraints
        if mol and satisfies_saturates_constraints(mol):
            generated_mols[sara_type].append(smiles)

